# 05b -- Hard Negative Mining (EfficientNetB0)

**Purpose:** Run the full HNM pipeline for EfficientNetB0 in three modes:
1. **Percentile-based** threshold (tau_mode=percentile) -- recommended default
2. **Threshold sweep** (tau_mode=sweep) -- ablation over multiple tau values
3. **Extended training without HNM** (--no_injection) -- critical control experiment

**Compute:** Colab T4 GPU (required).

**Estimated runtime:** ~60-90 minutes total (all three modes).

**Script:** `scripts/train_hnm.py`

**Note:** Replace `PATH` below with the actual baseline model checkpoint path
(e.g., `/content/drive/MyDrive/models/efficientnet_baseline_TIMESTAMP.keras`).

In [ ]:
# Mount Google Drive and verify GPU runtime
from google.colab import drive
drive.mount('/content/drive')

import subprocess
gpu_info = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if 'T4' in gpu_info.stdout:
    print('Runtime: Colab T4 GPU')
elif 'failed' in gpu_info.stderr.lower() or gpu_info.returncode != 0:
    print('Runtime: CPU only -- switch to a GPU runtime!')
else:
    print(f'Runtime: GPU detected\n{gpu_info.stdout[:200]}')

In [ ]:
%cd /content/drive/MyDrive/imagevalidation2

In [ ]:
# Mode 1: Percentile-based threshold (default)
!python scripts/train_hnm.py \
    --arch efficientnet \
    --model_path /content/drive/MyDrive/models/efficientnet_baseline_TIMESTAMP.keras \
    --data_dir /content/drive/MyDrive/FloodingDataset2 \
    --output_dir /content/drive/MyDrive/models \
    --tau_mode percentile

In [ ]:
# Mode 2: Threshold sweep (ablation over multiple tau values)
!python scripts/train_hnm.py \
    --arch efficientnet \
    --model_path /content/drive/MyDrive/models/efficientnet_baseline_TIMESTAMP.keras \
    --data_dir /content/drive/MyDrive/FloodingDataset2 \
    --output_dir /content/drive/MyDrive/models \
    --tau_mode sweep

In [ ]:
# Mode 3: Extended training WITHOUT HNM injection (critical control)
# This isolates the HNM contribution from the benefit of additional training epochs.
!python scripts/train_hnm.py \
    --arch efficientnet \
    --model_path /content/drive/MyDrive/models/efficientnet_baseline_TIMESTAMP.keras \
    --data_dir /content/drive/MyDrive/FloodingDataset2 \
    --output_dir /content/drive/MyDrive/models \
    --no_injection

In [ ]:
# List saved HNM models
import os
model_dir = '/content/drive/MyDrive/models'
if os.path.exists(model_dir):
    for f in sorted(os.listdir(model_dir)):
        if 'efficientnet' in f.lower() and ('hnm' in f.lower() or 'extended' in f.lower()):
            print(f)